In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Load the dataset 
# (Note: Update 'HR_comma_sep.csv' to match your actual dataset filename if needed)
df = pd.read_csv('HR_comma_sep.csv')

print("=== 1. Dataset Overview ===")
print("First 5 rows:")
display(df.head()) # Use print(df.head()) if running in a standard terminal

print("\nDataset Shape (Rows, Columns):", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Value Counts:")
print(df.isnull().sum())

print("\nTarget Class Counts ('left'):")
print(df['left'].value_counts())

# 2. Set 'left' as the target and remaining columns as features
X = df.drop(columns=['left'])
y = df['left']

# 3. Split the dataset into 70% training and 30% testing (random_state=25)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=25
)

# 4. Setup ColumnTransformer for OneHotEncoder and keep numerical features unchanged
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Keeps numerical features unchanged
)

# 5. Fit transformer on training data only, then transform both training and test sets
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("\nData preparation complete. Training shape:", X_train_transformed.shape)
print("Testing shape:", X_test_transformed.shape)

=== 1. Dataset Overview ===
First 5 rows:


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low



Dataset Shape (Rows, Columns): (14995, 10)

Data Types:
satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
left                       int64
promotion_last_5years      int64
Department                   str
salary                       str
dtype: object

Missing Value Counts:
satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
Work_accident            0
left                     0
promotion_last_5years    0
Department               0
salary                   0
dtype: int64

Target Class Counts ('left'):
left
0    11428
1     3567
Name: count, dtype: int64

Data preparation complete. Training shape: (10496, 18)
Testing shape: (4499, 18)


C:\Users\HP\AppData\Local\Temp\ipykernel_14020\4121064776.py:35: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import classification_report

# ==========================================
# 6. Create the Base Estimators
# ==========================================
lr_base = LogisticRegression(random_state=25, max_iter=1000)
dt_base = DecisionTreeClassifier(random_state=25)
knn_base = KNeighborsClassifier()
gnb_base = GaussianNB()

# ==========================================
# 7 & 8. Train Bagging Classifiers & Evaluate
# ==========================================
# Dictionary of the requested base estimators for bagging
bagging_estimators = {
    'Bagging + Decision Tree': dt_base,
    'Bagging + KNN': knn_base,
    'Bagging + Gaussian NB': gnb_base
}

# Train each Bagging model, predict, and display classification reports
for name, base_est in bagging_estimators.items():
    # Initialize BaggingClassifier with n_estimators=10 and random_state=25
    bagging_model = BaggingClassifier(
        estimator=base_est, 
        n_estimators=10, 
        random_state=25
    )
    
    # Fit on training data
    bagging_model.fit(X_train_transformed, y_train)
    
    # Predict on test data
    y_pred = bagging_model.predict(X_test_transformed)
    
    # Display classification report
    print(f"=== Classification Report: {name} ===")
    print(classification_report(y_test, y_pred))
    print("=" * 60 + "\n")

=== Classification Report: Bagging + Decision Tree ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3407
           1       0.99      0.97      0.98      1092

    accuracy                           0.99      4499
   macro avg       0.99      0.98      0.99      4499
weighted avg       0.99      0.99      0.99      4499


=== Classification Report: Bagging + KNN ===
              precision    recall  f1-score   support

           0       0.98      0.95      0.96      3407
           1       0.85      0.93      0.89      1092

    accuracy                           0.94      4499
   macro avg       0.92      0.94      0.93      4499
weighted avg       0.95      0.94      0.95      4499


=== Classification Report: Bagging + Gaussian NB ===
              precision    recall  f1-score   support

           0       0.90      0.71      0.80      3407
           1       0.46      0.76      0.57      1092

    accuracy               

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import log_loss
import pandas as pd

# 10. Define base estimators and ensemble sizes
base_estimators = {
    'Logistic Regression': LogisticRegression(random_state=25, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=25),
    'KNN': KNeighborsClassifier(),
    'GaussianNB': GaussianNB()
}

n_estimators_list = [10, 15, 25, 50]

results_c = []

# Loop through every combination (4 base estimators * 4 ensemble sizes = 16 configurations)
for name, base_est in base_estimators.items():
    for n_est in n_estimators_list:
        # Initialize BaggingClassifier
        bagging_model = BaggingClassifier(
            estimator=base_est,
            n_estimators=n_est,
            random_state=25
        )
        
        # Fit on training data
        bagging_model.fit(X_train_transformed, y_train)
        
        # 11. Predict test-set class probabilities
        y_prob = bagging_model.predict_proba(X_test_transformed)
        
        # Calculate Log Loss
        loss = log_loss(y_test, y_prob)
        
        # 12. Store configuration and Log Loss
        results_c.append({
            'Base Estimator': name,
            'Number of Estimators': n_est,
            'Log Loss': loss
        })

# Convert results into a DataFrame
df_results_c = pd.DataFrame(results_c)

# 13. Sort from lowest to highest Log Loss (lower is better) and display top five
df_sorted_c = df_results_c.sort_values(by='Log Loss', ascending=True).reset_index(drop=True)

print("=== Top 5 Configurations (Lowest Log Loss) ===")
print(df_sorted_c.head(5).to_string(index=False))

# 14. Identify the configuration with the lowest observed test Log Loss
best_config = df_sorted_c.iloc[0]
print("\n=== Configuration with Lowest Test Log Loss ===")
print(f"Base Estimator: {best_config['Base Estimator']}")
print(f"Number of Estimators: {best_config['Number of Estimators']}")
print(f"Log Loss: {best_config['Log Loss']:.4f}")

=== Top 5 Configurations (Lowest Log Loss) ===
Base Estimator  Number of Estimators  Log Loss
 Decision Tree                    50  0.154963
 Decision Tree                    25  0.162678
 Decision Tree                    15  0.200106
 Decision Tree                    10  0.221365
           KNN                    50  0.398799

=== Configuration with Lowest Test Log Loss ===
Base Estimator: Decision Tree
Number of Estimators: 50
Log Loss: 0.1550


In [5]:


# 15. Define depth settings for the Decision Tree base estimator
max_depth_values = [None, 3, 5, 7]
results_d = []

# Loop through each depth setting
for depth in max_depth_values:
    # Initialize base estimator with current max_depth and random_state=25
    base_tree = DecisionTreeClassifier(max_depth=depth, random_state=25)
    
    # Initialize BaggingClassifier with n_estimators=50 and random_state=25
    bagging_model = BaggingClassifier(
        estimator=base_tree,
        n_estimators=50,
        random_state=25
    )
    
    # Fit on training data
    bagging_model.fit(X_train_transformed, y_train)
    
    # 16. Predict test-set class probabilities and calculate Log Loss
    y_prob = bagging_model.predict_proba(X_test_transformed)
    loss = log_loss(y_test, y_prob)
    
    # Store results
    results_d.append({
        'Base Tree Max Depth': str(depth),  # Convert None to string for clean formatting
        'Log Loss': loss
    })

# 17. Present results in a DataFrame sorted in ascending order of Log Loss
df_results_d = pd.DataFrame(results_d)
df_sorted_d = df_results_d.sort_values(by='Log Loss', ascending=True).reset_index(drop=True)

print("=== Part D: Decision Tree Depth Experiment Results ===")
print(df_sorted_d.to_string(index=False))

=== Part D: Decision Tree Depth Experiment Results ===
Base Tree Max Depth  Log Loss
                  7  0.077625
                  5  0.087425
                  3  0.142670
               None  0.154963


In [8]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [9]:


# 19. Train a Bagging model with OOB scoring enabled
bagging_oob = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None, random_state=25),
    n_estimators=25,
    random_state=25,
    bootstrap=True,
    oob_score=True
)

bagging_oob.fit(X_train_transformed, y_train)

# 20. Display the model’s OOB score
oob_score = bagging_oob.oob_score_
print(f"=== Out-of-Bag (OOB) Accuracy Score: {oob_score:.4f} ===")

# 21. Predict the test labels and display test accuracy and classification report
y_pred_test = bagging_oob.predict(X_test_transformed)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Test Accuracy Score: {test_accuracy:.4f}\n")
print("=== Classification Report (Test Set) ===")
print(classification_report(y_test, y_pred_test))

=== Out-of-Bag (OOB) Accuracy Score: 0.9859 ===
Test Accuracy Score: 0.9891

=== Classification Report (Test Set) ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3407
           1       0.99      0.97      0.98      1092

    accuracy                           0.99      4499
   macro avg       0.99      0.98      0.99      4499
weighted avg       0.99      0.99      0.99      4499

